# Parte 3 — Preparación de datos (IE2)
## Ames Iowa Housing Dataset — Proyecto MLY1101, Caso B

Este bloque continúa directamente desde el EDA de la Parte 2. Decisiones acordadas
por el equipo antes de programar:

1. **`PID 908154235`** (Gr Liv Area = 5642, Sale Condition = Partial, SalePrice = 160.000) → **se elimina**.
   Justificación: De Cock (2011), autor del dataset, identifica esta observación específica
   como una venta parcial no representativa del valor real de mercado. Los otros dos casos
   `Partial` del grupo >4000 ft² (PID 908154195 y 908154205) **se mantienen**, ya que corresponden
   a precios coherentes con viviendas grandes genuinas.
2. **`Neighborhood`** → se agrupan en `'Other'` las categorías con menos de 20 casos
   (`Blueste`, `Greens`, `GrnHill`, `Landmrk`), luego se aplica one-hot encoding.
3. **`SalePrice`** → no se transforma con `log1p` en esta etapa; queda documentada como
   propuesta para la fase de Modelamiento (fuera del alcance de Parte 3 / IE2).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/Ames_Iowa_Housing_Dataset.csv')
print(f"Shape inicial: {df.shape}")

Shape inicial: (2930, 82)


## 1. Corrección de errores y anomalías puntuales detectadas en el EDA

- `Garage Yr Blt = 2207` (PID 916384070): error de tipeo evidente, se corrige a 2007
  (coincide con `Year Remod/Add`).
- Se elimina el registro `PID 908154235` por la razón explicada arriba.

In [ ]:
# Corregir año de garaje invalido
mask_typo = df['Garage Yr Blt'] > 2024
assert mask_typo.sum() == 1
df.loc[mask_typo, 'Garage Yr Blt'] = 2007
print("Corregido: Garage Yr Blt 2207 -> 2007 (PID 916384070)")

# Eliminar unicamente el registro de venta parcial no representativa
before = len(df)
df = df[df['PID'] != 908154235].copy()
print(f"Eliminado(s) {before - len(df)} registro(s): PID 908154235")
print(f"Shape tras limpieza puntual: {df.shape}")

# Verificacion: los otros 2 casos Partial >4000 ft2 se mantienen
partial_kept = df[(df['Gr Liv Area'] > 4000) & (df['Sale Condition'] == 'Partial')]
print(f"Casos Partial >4000 ft2 mantenidos intencionalmente: {len(partial_kept)}")

Corregido: Garage Yr Blt 2207 -> 2007 (PID 916384070)
Eliminado(s) 1 registro(s): PID 908154235
Shape tras limpieza puntual: (2929, 82)
Casos Partial >4000 ft2 mantenidos intencionalmente: 2


## 2. Imputación de valores faltantes

El diccionario oficial del dataset (De Cock, 2011) confirma que la mayoría de los nulos
corresponden a **ausencia física** del atributo (sin sótano, sin garaje, sin piscina, etc.),
no a datos perdidos por error. Se tratan de forma diferenciada:

- **Ausencia física** (categóricas): se imputa `'None'`
- **Ausencia física** (numéricas asociadas): se imputa `0`
- **`Lot Frontage`**: mediana agrupada por `Neighborhood` (con respaldo de mediana global
  para los 2 barrios que no tienen ningún valor registrado)
- **`Electrical`**: moda (1 solo nulo aislado)
- **Casos borde**: 2 registros con `Garage Type` presente pero datos de garaje incompletos
  (`PID 910201180` y `PID 903426160`) — se imputan con la mediana/moda del mismo tipo de
  garaje, no con `'None'`, porque sabemos que el garaje sí existe.

In [3]:
# Bloque de ausencia fisica (categoricas -> 'None')
cat_none_cols = ['Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Mas Vnr Type',
                  'Fireplace Qu', 'Garage Type', 'Bsmt Qual', 'Bsmt Cond',
                  'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2']
for col in cat_none_cols:
    df[col] = df[col].fillna('None')

# Garage Finish / Qual / Cond: 'None' salvo casos borde con garaje real
for col in ['Garage Finish', 'Garage Qual', 'Garage Cond']:
    has_garage_null = df[col].isnull() & (df['Garage Type'] != 'None')
    if has_garage_null.any():
        for gtype in df.loc[has_garage_null, 'Garage Type'].unique():
            mode_val = df.loc[(df['Garage Type'] == gtype) & df[col].notna(), col].mode()
            fill_val = mode_val[0] if len(mode_val) else 'TA'
            m = has_garage_null & (df['Garage Type'] == gtype)
            df.loc[m, col] = fill_val
    df[col] = df[col].fillna('None')

# Garage Yr Blt: 0 sin garaje, Year Built si el garaje existe pero falta el dato
no_garage = df['Garage Type'] == 'None'
df.loc[no_garage, 'Garage Yr Blt'] = df.loc[no_garage, 'Garage Yr Blt'].fillna(0)
has_garage_yr_null = df['Garage Yr Blt'].isnull() & (~no_garage)
df.loc[has_garage_yr_null, 'Garage Yr Blt'] = df.loc[has_garage_yr_null, 'Year Built']

# Garage Cars / Garage Area: 0 sin garaje, mediana del mismo tipo si el garaje existe
for col in ['Garage Cars', 'Garage Area']:
    needs_fill = df[col].isnull() & (~no_garage)
    if needs_fill.any():
        for gtype in df.loc[needs_fill, 'Garage Type'].unique():
            med = df.loc[(df['Garage Type'] == gtype), col].median()
            m = needs_fill & (df['Garage Type'] == gtype)
            df.loc[m, col] = med
    df[col] = df[col].fillna(0)

# Mas Vnr Area y variables numericas de sotano: 0 (ausencia fisica)
df['Mas Vnr Area'] = df['Mas Vnr Area'].fillna(0)
bsmt_num_cols = ['Bsmt Full Bath', 'Bsmt Half Bath', 'BsmtFin SF 1', 'BsmtFin SF 2',
                  'Bsmt Unf SF', 'Total Bsmt SF']
for col in bsmt_num_cols:
    df[col] = df[col].fillna(0)

# Lot Frontage: mediana por Neighborhood + respaldo de mediana global
global_median_lf = df['Lot Frontage'].median()
df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(lambda x: x.fillna(x.median()))
df['Lot Frontage'] = df['Lot Frontage'].fillna(global_median_lf)

# Electrical: moda
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

print(f"Nulos restantes: {df.isnull().sum().sum()}")

Nulos restantes: 0


## 3. Tratamiento de outliers

Se mantiene la conclusión del EDA de Parte 2: los outliers de `SalePrice` y `Lot Frontage`
(método IQR) corresponden mayoritariamente a viviendas reales de alto valor, no a errores
de carga. **No se eliminan en bloque** — solo se documentan para que la fase de Modelamiento
decida si aplica regularización o transformación.

In [4]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

low, high = iqr_bounds(df['SalePrice'])
n_out = ((df['SalePrice'] < low) | (df['SalePrice'] > high)).sum()
print(f"SalePrice - outliers IQR: {n_out} ({n_out/len(df)*100:.2f}%) -> se mantienen, documentado")

low_lf, high_lf = iqr_bounds(df['Lot Frontage'])
n_out_lf = ((df['Lot Frontage'] < low_lf) | (df['Lot Frontage'] > high_lf)).sum()
print(f"Lot Frontage - outliers IQR: {n_out_lf} ({n_out_lf/len(df)*100:.2f}%) -> se mantienen, documentado")

SalePrice - outliers IQR: 137 (4.68%) -> se mantienen, documentado
Lot Frontage - outliers IQR: 205 (7.00%) -> se mantienen, documentado


## 4. Ingeniería de variables

- Se reemplazan los años absolutos por variables de **edad** (`House_Age`, `Remod_Age`,
  `Garage_Age`), más interpretables para el modelo.
- `Total_SF` combina `Total Bsmt SF + 1st Flr SF + 2nd Flr SF` para mitigar la
  multicolinealidad detectada en el EDA (`Total Bsmt SF` ↔ `1st Flr SF`, r=0.80).
- Se elimina `Garage Area` del set de modelado y se conserva `Garage Cars`
  (r=0.89 entre ambas — redundantes; se prioriza la variable discreta, más fácil
  de explicar en la defensa).

In [5]:
df['House_Age'] = (df['Yr Sold'] - df['Year Built']).clip(lower=0)
df['Remod_Age'] = (df['Yr Sold'] - df['Year Remod/Add']).clip(lower=0)
df['Garage_Age'] = np.where(df['Garage Yr Blt'] == 0, 0, df['Yr Sold'] - df['Garage Yr Blt']).clip(min=0)
df['Total_SF'] = df['Total Bsmt SF'] + df['1st Flr SF'] + df['2nd Flr SF']

cols_to_drop = ['Garage Area', 'Year Built', 'Year Remod/Add', 'Garage Yr Blt']
df_model = df.drop(columns=cols_to_drop)
print(f"Variables nuevas: House_Age, Remod_Age, Garage_Age, Total_SF")
print(f"Variables eliminadas por redundancia: {cols_to_drop}")
print(f"Shape: {df_model.shape}")

Variables nuevas: House_Age, Remod_Age, Garage_Age, Total_SF
Variables eliminadas por redundancia: ['Garage Area', 'Year Built', 'Year Remod/Add', 'Garage Yr Blt']
Shape: (2929, 82)


## 5. Encoding

- **23 variables ordinales** (según ficha oficial De Cock, 2011): mapeo manual a escala
  numérica, preservando el orden (ej. `Kitchen Qual`: Po=1 < Fa=2 < TA=3 < Gd=4 < Ex=5).
- **`Neighborhood`**: categorías con <20 casos agrupadas en `'Other'`, luego one-hot.
- **`MS SubClass`**: se fuerza a tipo categórico antes de encodear — es nominal aunque
  esté codificada con números.
- **Resto de nominales**: one-hot encoding estándar.
- **Identificadores** (`PID`, `Order`): se retiran del set de features, no se modelan.

In [6]:
qual_scale = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

ordinal_maps = {
    'Lot Shape':      {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3},
    'Utilities':      {'ELO': 0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3},
    'Land Slope':     {'Sev': 0, 'Mod': 1, 'Gtl': 2},
    'Exter Qual': qual_scale, 'Exter Cond': qual_scale,
    'Bsmt Qual': qual_scale, 'Bsmt Cond': qual_scale,
    'Bsmt Exposure':  {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFin Type 1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFin Type 2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'Heating QC': qual_scale,
    'Electrical':     {'FuseP': 0, 'FuseF': 1, 'FuseA': 2, 'Mix': 3, 'SBrkr': 4},
    'Kitchen Qual': qual_scale,
    'Functional':     {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7},
    'Fireplace Qu': qual_scale,
    'Garage Finish':  {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'Garage Qual': qual_scale, 'Garage Cond': qual_scale,
    'Paved Drive':    {'N': 0, 'P': 1, 'Y': 2},
    'Pool QC':        {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
    'Fence':          {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
}

for col, mapping in ordinal_maps.items():
    unseen = set(df_model[col].dropna().unique()) - set(mapping.keys())
    assert not unseen, f"Categorias no contempladas en {col}: {unseen}"
    df_model[col] = df_model[col].map(mapping)

print("Encoding ordinal aplicado (23 variables) sin categorias huerfanas")

# Neighborhood: agrupar categorias raras
nb_counts = df_model['Neighborhood'].value_counts()
rare = nb_counts[nb_counts < 20].index.tolist()
df_model['Neighborhood'] = df_model['Neighborhood'].replace(rare, 'Other')
print(f"Neighborhood: {len(rare)} categorias agrupadas en 'Other': {rare}")

# MS SubClass es nominal aunque parezca numerica
df_model['MS SubClass'] = df_model['MS SubClass'].astype(str)

nominal_cols = ['MS SubClass', 'MS Zoning', 'Street', 'Alley', 'Land Contour',
                 'Lot Config', 'Neighborhood', 'Condition 1', 'Condition 2',
                 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
                 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Foundation',
                 'Heating', 'Central Air', 'Garage Type', 'Misc Feature',
                 'Sale Type', 'Sale Condition']
nominal_cols = [c for c in nominal_cols if c in df_model.columns]
df_model = pd.get_dummies(df_model, columns=nominal_cols, drop_first=True)

df_model = df_model.drop(columns=['PID', 'Order'], errors='ignore')

print(f"Nulos finales: {df_model.isnull().sum().sum()}")
print(f"Shape final (listo para modelar): {df_model.shape}")

Encoding ordinal aplicado (23 variables) sin categorias huerfanas
Neighborhood: 4 categorias agrupadas en 'Other': ['Blueste', 'Greens', 'GrnHill', 'Landmrk']
Nulos finales: 0
Shape final (listo para modelar): (2929, 220)


## 6. Exportar resultados

Se generan dos archivos:
- **Versión legible** (post-limpieza, pre one-hot): útil para seguir documentando el
  README y para inspección visual del equipo.
- **Versión final codificada**: lista para la fase de Modelamiento.

In [7]:
df.to_csv('../data/Ames_Housing_Parte3_legible.csv', index=False)
df_model.to_csv('../data/Ames_Housing_Parte3_listo_para_modelar.csv', index=False)
print("Archivos exportados.")
print(f" - Version legible: {df.shape}")
print(f" - Version lista para modelar: {df_model.shape}")

Archivos exportados.
 - Version legible: (2929, 86)
 - Version lista para modelar: (2929, 220)
